# Clase 13 · Método de selección de variables: práctica guiada (baseline)

En la clase se usa el dataset de Kaggle *mobile price classification*: 20 variables numéricas y `price_range` con 4 niveles (0–3).
El repo no lo tiene, así que aquí construimos una **versión sintética** con las mismas 20 columnas (📌 Extra).
En ella **solo 4 variables influyen** en el precio y las otras 16 son ruido.
También usamos `data/raw/clientes_banco.csv` para ver por qué importa `stratify` y por qué el accuracy engaña.

Pasos: (1) cada fila es un punto en $\mathbb{R}^{20}$ · (2) división estratificada · (3) Random Forest podado = **baseline** · (4) accuracy y matriz de confusión · (5) ¿se puede mejorar con menos variables?

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split

# Busca la carpeta raíz del repo (la que contiene data/)
RAIZ = Path.cwd()
while not (RAIZ / "data" / "generar_datos.py").exists() and RAIZ != RAIZ.parent:
    RAIZ = RAIZ.parent
banco = pd.read_csv(RAIZ / "data" / "raw" / "clientes_banco.csv")


def generar_moviles(n=2000, semilla=13):
    """Versión sintética de 'mobile price classification' (la misma que usan los tests)."""
    rng = np.random.default_rng(semilla)
    d = pd.DataFrame({
        "battery_power": rng.integers(500, 2000, n), "blue": rng.integers(0, 2, n),
        "clock_speed": rng.uniform(0.5, 3.0, n).round(1), "dual_sim": rng.integers(0, 2, n),
        "fc": rng.integers(0, 20, n), "four_g": rng.integers(0, 2, n),
        "int_memory": rng.integers(2, 65, n), "m_dep": rng.uniform(0.1, 1.0, n).round(1),
        "mobile_wt": rng.integers(80, 201, n), "n_cores": rng.integers(1, 9, n),
        "pc": rng.integers(0, 21, n), "px_height": rng.integers(0, 1961, n),
        "px_width": rng.integers(500, 2000, n), "ram": rng.integers(256, 4000, n),
        "sc_h": rng.integers(5, 20, n), "sc_w": rng.integers(0, 19, n),
        "talk_time": rng.integers(2, 21, n), "three_g": rng.integers(0, 2, n),
        "touch_screen": rng.integers(0, 2, n), "wifi": rng.integers(0, 2, n),
    })
    z = lambda c: (d[c] - d[c].mean()) / d[c].std()
    puntaje = 3 * z("ram") + z("battery_power") + 0.7 * z("px_height") + 0.7 * z("px_width") + rng.normal(0, 0.5, n)
    d["price_range"] = pd.qcut(puntaje, 4, labels=False).astype(int)
    return d


data = generar_moviles()
data.head()

## 1. Cada fila es un punto en $\mathbb{R}^{20}$
Como en la clase, `data.info()` muestra que **todo es numérico y no hay nulos**. Así podemos concentrarnos solo en el efecto de las variables, sin imputar ni escalar.

In [ ]:
data.info()

In [ ]:
X = data.drop(columns="price_range")   # predictores
y = data["price_range"]                # objetivo (4 clases)
print("Dimensión de cada muestra: R^", X.shape[1])
print("Un móvil = un punto:", X.iloc[0].to_dict())
y.value_counts().sort_index()

## 2. División train/test con `stratify=y`
*"Trata de forma proporcionada de extraer de cada categoría cierta cantidad de muestras."*

En los móviles las 4 clases están balanceadas (500 cada una), así que el efecto se nota poco. En `clientes_banco.csv` solo ~5 % abandona: ahí sí importa.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=1)
print(len(X_train), "train /", len(X_test), "test")
print("Clases en test:", y_test.value_counts().sort_index().to_dict())

In [ ]:
yb = banco["abandono"]
Xb = banco.drop(columns=["abandono", "cliente_id", "segmento"])
print(f"Tasa de abandono total: {yb.mean():.4f}  ({yb.sum()} de {len(yb)})")

for semilla in [1, 2, 3]:
    _, _, _, yte_sin = train_test_split(Xb, yb, test_size=0.2, random_state=semilla)
    _, _, _, yte_con = train_test_split(Xb, yb, test_size=0.2, random_state=semilla, stratify=yb)
    print(f"semilla {semilla}: positivos en test SIN stratify = {yte_sin.sum():3d} | CON stratify = {yte_con.sum():3d}")

Con `stratify`, el test siempre tiene ~20 % de los positivos (51). Sin él, la cantidad cambia con la semilla.

## 3. El baseline: Random Forest "podado"
- **Profundidad** (`max_depth=5`): el árbol crece como máximo 5 niveles.
- **Nodos hoja** (`max_leaf_nodes=20`): como máximo 20 nodos finales por árbol.
- Un solo árbol sobreajusta rápido. El bosque usa 100 árboles (por defecto) y se **poda** cada uno.

In [ ]:
modelo = RandomForestClassifier(max_depth=5, max_leaf_nodes=20, random_state=1)
modelo.fit(X_train, y_train)
y_pred = modelo.predict(X_test)
acc_baseline = accuracy_score(y_test, y_pred)
print(f"Accuracy baseline (20 variables): {acc_baseline:.4f}")

La clase obtuvo ≈ 0.83 con el dataset real de Kaggle. En el sintético da ≈ 0.75. Son datos distintos: lo importante es tener **un número de referencia**.

## 4. Matriz de confusión
Filas = clase **real**, columnas = clase **predicha**. La **diagonal** son los aciertos.

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=[0, 1, 2, 3])
pd.DataFrame(cm, index=[f"real {c}" for c in range(4)], columns=[f"pred {c}" for c in range(4)])

In [ ]:
# En la clase se graficó con ConfusionMatrixDisplay (necesita matplotlib).
try:
    import matplotlib  # noqa: F401
    from sklearn.metrics import ConfusionMatrixDisplay
    ConfusionMatrixDisplay(cm).plot()
except ImportError:
    print("matplotlib no está instalado: usamos la tabla de arriba.")

### ✏️ Tu turno 1
Calcula el accuracy **solo a partir de `cm`**, sin usar `y_test` ni `y_pred`.
Pista: la diagonal son los aciertos (`np.trace`) y la suma de toda la matriz es el total.
Comprueba que da lo mismo que `acc_baseline`. ¿Qué par de clases se confunde más?

In [ ]:
# tu código aquí


## 5. Cuidado: el accuracy engaña con datos desbalanceados
Un "modelo" que **siempre** dice "no abandona" en `clientes_banco.csv`:

In [ ]:
pred_tonta = np.zeros(len(yb), dtype=int)
print(f"Accuracy prediciendo siempre 0: {accuracy_score(yb, pred_tonta):.4f}")
print("Clientes que abandonan detectados:", int(((pred_tonta == 1) & (yb == 1)).sum()), "de", int(yb.sum()))
print(confusion_matrix(yb, pred_tonta))

≈ 0.95 de accuracy y **cero** abandonos detectados. La matriz de confusión lo delata: la columna "pred 1" está vacía.

### ✏️ Tu turno 2
Divide `Xb, yb` **sin** `stratify` con 10 semillas distintas (0 a 9) y guarda la tasa de abandono del test en cada una.
¿Cuál es la mínima y cuál la máxima? Repite **con** `stratify=yb`. ¿Qué cambia?

In [ ]:
# tu código aquí


## 6. ¿Se puede mejorar con menos variables?
*"Si este fue mi primer resultado, ¿será que lo puedo mejorar o no?"*

Aquí hacemos trampa a propósito: como el dataset es inventado, **sabemos** qué 4 variables importan. En la vida real hay que descubrirlas, y eso es lo que enseñan las clases 14–17.

In [ ]:
def accuracy_con(columnas):
    m = RandomForestClassifier(max_depth=5, max_leaf_nodes=20, random_state=1)
    m.fit(X_train[columnas], y_train)
    return accuracy_score(y_test, m.predict(X_test[columnas]))

utiles = ["ram", "battery_power", "px_height", "px_width"]
ruido = ["blue", "wifi", "sc_h", "talk_time"]
print(f"Todas (R^20):     {accuracy_con(list(X.columns)):.4f}")
print(f"4 útiles (R^4):   {accuracy_con(utiles):.4f}")
print(f"4 de ruido (R^4): {accuracy_con(ruido):.4f}   <- casi azar (0.25 con 4 clases)")
print(f"solo ram (R^1):   {accuracy_con(['ram']):.4f}")

Menos variables, **pero las correctas**, dan un modelo mejor y más simple. Con las 4 equivocadas, el modelo queda casi al azar.

### ✏️ Tu turno 3
1. Prueba `accuracy_con(["ram", "battery_power"])`. ¿Cuánto pierdes respecto de las 4 útiles?
2. Cambia `max_depth` a 2 y luego a 15 en `accuracy_con` (con todas las variables). ¿Qué pasa con el accuracy? Relaciónalo con la "poda".
3. Implementa las funciones de `ejercicio.py` y evalúa con
   `python -m framework evaluar cursos/nivelacion/n1-python-data-science/m3-seleccion-variables/clase-13-seleccion`.

In [ ]:
# tu código aquí
